In [1]:
!pip install -q datasets

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

from datasets import load_dataset
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using Device:", device)

Using Device: cpu


In [4]:
print("Loading AG News Dataset...")

dataset = load_dataset("fancyzhx/ag_news")

train_data = dataset["train"]
test_data = dataset["test"]

print("Training Samples :", len(train_data))
print("Testing Samples  :", len(test_data))

Loading AG News Dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Training Samples : 120000
Testing Samples  : 7600


In [5]:
print("Building Vocabulary...")

vocab = {
    "<unk>": 0,
    "<pad>": 1
}

def tokenize(text):
    return text.lower().split()

for article in train_data["text"]:
    words = tokenize(article)

    for word in words:
        if word not in vocab:
            vocab[word] = len(vocab)

print("Vocabulary Size:", len(vocab))

PAD_IDX = vocab["<pad>"]

Building Vocabulary...
Vocabulary Size: 158735


In [6]:
def numericalize(text):
    words = tokenize(text)

    return [
        vocab.get(word, vocab["<unk>"])
        for word in words
    ]

In [7]:
def collate_batch(batch):

    text_list = []
    label_list = []
    length_list = []

    for item in batch:

        text_tensor = torch.tensor(
            numericalize(item["text"]),
            dtype=torch.long
        )

        text_list.append(text_tensor)
        label_list.append(item["label"])
        length_list.append(len(text_tensor))

    padded_texts = pad_sequence(
        text_list,
        batch_first=True,
        padding_value=PAD_IDX
    )

    return (
        padded_texts.to(device),
        torch.tensor(label_list).to(device),
        torch.tensor(length_list)
    )

In [8]:
BATCH_SIZE = 64

train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch
)

test_loader = DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch
)

In [9]:
class SimpleRNNClassifier(nn.Module):

    def __init__(self, vocab_size):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            100,
            padding_idx=PAD_IDX
        )

        self.rnn = nn.RNN(
            input_size=100,
            hidden_size=128,
            batch_first=True
        )

        self.fc = nn.Linear(
            128,
            4
        )

    def forward(self, text, lengths):

        embedded = self.embedding(text)

        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _, hidden = self.rnn(packed)

        hidden = hidden.squeeze(0)

        output = self.fc(hidden)

        return output

In [10]:
model = SimpleRNNClassifier(
    vocab_size=len(vocab)
).to(device)

print(model)

SimpleRNNClassifier(
  (embedding): Embedding(158735, 100, padding_idx=1)
  (rnn): RNN(100, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=4, bias=True)
)


In [11]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)


In [12]:
def train():

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for texts, labels, lengths in train_loader:

        optimizer.zero_grad()

        outputs = model(texts, lengths)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()

        total += labels.size(0)

    accuracy = 100 * correct / total

    return total_loss, accuracy

In [13]:
def evaluate():

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for texts, labels, lengths in test_loader:

            outputs = model(texts, lengths)

            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()

            total += labels.size(0)

    accuracy = 100 * correct / total

    return accuracy

In [14]:
EPOCHS = 5

print("\nStarting Training...\n")

for epoch in range(EPOCHS):

    train_loss, train_acc = train()

    test_acc = evaluate()

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Loss: {train_loss:.4f} "
        f"Train Accuracy: {train_acc:.2f}% "
        f"Test Accuracy: {test_acc:.2f}%"
    )



Starting Training...

Epoch [1/5] Loss: 1849.8642 Train Accuracy: 56.48% Test Accuracy: 59.80%
Epoch [2/5] Loss: 1603.8521 Train Accuracy: 64.26% Test Accuracy: 71.04%
Epoch [3/5] Loss: 1255.7319 Train Accuracy: 73.62% Test Accuracy: 69.75%
Epoch [4/5] Loss: 1174.0208 Train Accuracy: 75.34% Test Accuracy: 63.64%
Epoch [5/5] Loss: 888.3878 Train Accuracy: 82.57% Test Accuracy: 81.08%


In [15]:
torch.save(
    model.state_dict(),
    "ag_news_rnn.pth"
)

print("\nModel Saved Successfully!")


Model Saved Successfully!


In [16]:
categories = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

def predict_news(text):

    model.eval()

    tokens = torch.tensor(
        [numericalize(text)],
        dtype=torch.long
    ).to(device)

    lengths = torch.tensor(
        [tokens.size(1)]
    )

    with torch.no_grad():

        output = model(tokens, lengths)

        prediction = output.argmax(dim=1).item()

    return categories[prediction]

In [17]:
sample_news = """
Apple introduced a new AI-powered smartphone
with advanced machine learning features.
"""

prediction = predict_news(sample_news)

print("\nSample News:")
print(sample_news)

print("\nPredicted Category:", prediction)


Sample News:

Apple introduced a new AI-powered smartphone
with advanced machine learning features.


Predicted Category: Sci/Tech
